# Tennis Club
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Greedy, Heaps, Sorting · **Difficulty/Frequency:** Rare (2/10)


## Concepts

**What this problem is really testing:**
- Interval partitioning — the minimum number of "resources" needed to schedule intervals so none overlap on the same resource
- Solved greedily, using a min-heap
- A **greedy exchange argument** — a way to *prove* the greedy choice is actually optimal, not just "seems to work"

**Why it applies here:**
- "Assign each booking to a court, using the fewest courts, so no court double-books" is the textbook interval-partitioning problem.
- The key fact: **the minimum number of courts needed equals the maximum number of bookings overlapping at any single instant.** You genuinely can't do better than that — each of those overlapping bookings needs its own court right then — and a greedy assignment hits that number exactly.

**The one idea to hold onto:** process bookings in start-time order. For each one, reuse whichever existing court frees up *earliest*, if it's already free by this booking's start time. Only open a new court if none is free yet. A min-heap gives fast (O(log n)) access to "which court frees up soonest."

---

### Quick primers — the building blocks used below

**What is a Heap (priority queue)?**
- A heap keeps the min (or max) element accessible in O(1), with O(log n) push/pop, by keeping one rule true at all times: "parent ≤ children" (for a min-heap).
- Here, the heap holds `(available_time, court_id)` pairs. Peeking at the top always tells you which court becomes free soonest — exactly what the greedy rule needs to answer "can I reuse a court right now?"

**Greedy algorithms and the exchange argument.**
- A greedy algorithm always makes the locally-best choice at each step, with no backtracking.
- An **exchange argument** is how you *prove* a greedy choice is safe: show that if some optimal solution made a different choice at this step, you could swap it for the greedy choice without making the result any worse. That means an optimal solution exists that agrees with greedy here — and by repeating this logic, greedy turns out to be optimal all the way through.
- Applied here: if an optimal solution puts booking `i` on a court that frees up *later* than the earliest-available court, swapping to the earliest-available court can only help (it's free at least as soon). So always choosing the earliest-freeing court is never a worse move.

**Sorting to enable a single greedy pass.**
- Processing bookings in start-time order is what lets "is any court free right now?" be answered with a single heap-peek — instead of searching through every booking seen so far.


## Problem Statement

Given bookings with `(id, start_time, finish_time)`, assign each to a court such that no court is double-booked, using the **minimum number of courts** (unlimited courts available). Return the assignment (booking_id -> court_id pairs), not just the count.

**Part 2:** after each booking, a fixed maintenance time `X` is needed before the court can be reused.
**Part 3:** maintenance is needed only after `K` bookings on the same court (not after every single one).


### Part 1 -- Basic Court Assignment

**Idea:** sort bookings by start time. Maintain a min-heap of `(available_time, court_id)` for courts currently in use. For each booking (in start-time order): if the earliest-available court is free by this booking's start (`heap[0][0] <= start`), reuse it (pop, update its available time, push back); otherwise open a new court.

**Time complexity:** O(n log n) -- sorting once, then each of n bookings does O(log n) of heap work.

**Space complexity:** O(n) -- the heap holds at most one entry per court ever opened, bounded by n.


In [ ]:
import heapq
from typing import List, Tuple


class BookingRecord:
    def __init__(self, id: int, start_time: int, finish_time: int):
        self.Id = id
        self.Start_time = start_time
        self.Finish_time = finish_time


def assignCourts(bookingRecords: List[BookingRecord]) -> List[Tuple[int, int]]:
    bookings = sorted(bookingRecords, key=lambda b: b.Start_time)

    heap: List[Tuple[int, int]] = []      # (available_time, court_id)
    assignments: List[Tuple[int, int]] = []
    next_court_id = 0

    for booking in bookings:
        if heap and heap[0][0] <= booking.Start_time:      # earliest-freeing court is free by now
            _, court_id = heapq.heappop(heap)
            heapq.heappush(heap, (booking.Finish_time, court_id))
        else:                                                # no court free yet -- open a new one
            court_id = next_court_id
            next_court_id += 1
            heapq.heappush(heap, (booking.Finish_time, court_id))
        assignments.append((booking.Id, court_id))

    return assignments


def courts_used(assignments: List[Tuple[int, int]]) -> int:
    return len(set(court_id for _, court_id in assignments))


### Part 2 -- Maintenance After Every Booking

**Idea:** identical algorithm; the only change is what gets pushed back onto the heap: `finish_time + X` instead of `finish_time` -- a court isn't "available" until maintenance finishes too.


In [ ]:
def assignCourtsWithMaintenance(bookingRecords: List[BookingRecord], X: int) -> List[Tuple[int, int]]:
    bookings = sorted(bookingRecords, key=lambda b: b.Start_time)
    heap: List[Tuple[int, int]] = []
    assignments: List[Tuple[int, int]] = []
    next_court_id = 0

    for booking in bookings:
        if heap and heap[0][0] <= booking.Start_time:
            _, court_id = heapq.heappop(heap)
        else:
            court_id = next_court_id
            next_court_id += 1
        heapq.heappush(heap, (booking.Finish_time + X, court_id))   # +X: maintenance before reuse
        assignments.append((booking.Id, court_id))

    return assignments


### Part 3 -- Maintenance After Threshold Usage (every K bookings)

**Idea:** track a per-court usage counter. A court becomes available at `finish_time` as usual, UNLESS its usage count has reached `K` since the last maintenance, in which case it becomes available at `finish_time + X` and its counter resets to 0.

**Time complexity:** O(n log n) -- same heap operations, plus O(1) dict lookups per booking.


In [ ]:
from typing import Dict


def assignCourtsWithThresholdMaintenance(bookingRecords: List[BookingRecord], X: int, K: int) -> List[Tuple[int, int]]:
    bookings = sorted(bookingRecords, key=lambda b: b.Start_time)
    heap: List[Tuple[int, int]] = []
    court_usage: Dict[int, int] = {}
    assignments: List[Tuple[int, int]] = []
    next_court_id = 0

    for booking in bookings:
        if heap and heap[0][0] <= booking.Start_time:
            _, court_id = heapq.heappop(heap)
        else:
            court_id = next_court_id
            next_court_id += 1
            court_usage[court_id] = 0

        court_usage[court_id] = court_usage.get(court_id, 0) + 1
        if court_usage[court_id] >= K:
            available_time = booking.Finish_time + X
            court_usage[court_id] = 0                    # reset after maintenance
        else:
            available_time = booking.Finish_time

        heapq.heappush(heap, (available_time, court_id))
        assignments.append((booking.Id, court_id))

    return assignments


## Verification

Check against a hand-traceable example, the maintenance variants, and the optimality bound (min courts == max overlap).

In [ ]:
def make(id, s, f):
    return BookingRecord(id, s, f)


bookings = [make(1, 0, 10), make(2, 5, 15), make(3, 10, 20), make(4, 20, 30)]
# Overlap check by hand: [0,10) and [5,15) overlap on [5,10) -> need 2 courts then.
# Booking 3 starts at 10 == booking1's finish -- back-to-back reuse is allowed (heap[0][0] <= start).
# Booking 4 starts at 20 == booking3's finish -- also reusable.

assignments = assignCourts(bookings)
assert courts_used(assignments) == 2                 # max simultaneous overlap is 2 (bookings 1 & 2)
by_id = dict(assignments)
assert by_id[1] == by_id[3] or by_id[2] == by_id[3]   # booking 3 reuses whichever court freed at t=10
assert by_id[1] != by_id[2]                            # bookings 1 and 2 genuinely overlap -> different courts

# Back-to-back (touching) bookings on ONE court: start == previous finish should NOT need a new court
back_to_back = [make(1, 0, 5), make(2, 5, 10), make(3, 10, 15)]
assert courts_used(assignCourts(back_to_back)) == 1

# Fully overlapping triple -> needs exactly 3 courts
triple_overlap = [make(1, 0, 100), make(2, 0, 100), make(3, 0, 100)]
assert courts_used(assignCourts(triple_overlap)) == 3

# Part 2: maintenance forces a NEW court for back-to-back bookings that used to share one
maint = assignCourtsWithMaintenance(back_to_back, X=1)
assert courts_used(maint) == 2   # X=1 means finish+1 > next start (which equals finish) -> can't reuse immediately

# With X=0, maintenance changes nothing relative to Part 1
assert courts_used(assignCourtsWithMaintenance(back_to_back, X=0)) == 1

# Part 3: threshold maintenance -- K=2 means maintenance kicks in every 2nd booking on a court
four_sequential = [make(1, 0, 5), make(2, 5, 10), make(3, 10, 15), make(4, 15, 20)]
# All back-to-back with no gaps -- without maintenance, 1 court suffices (Part 1 baseline):
assert courts_used(assignCourts(four_sequential)) == 1
# With K=2, X=3: after every 2nd booking, +3 maintenance is added, which WILL cause a new court
# to be needed for booking 3 (starts at 10, but the court isn't free until finish(5)+3=8... still <=10,
# so actually still fine). Use a tighter gap to force the effect:
tight = [make(1, 0, 5), make(2, 5, 10), make(3, 10, 15), make(4, 15, 20)]
result_k2 = assignCourtsWithThresholdMaintenance(tight, X=10, K=2)
# Booking 2 triggers maintenance (2nd use of its court) -> court unavailable until 10+10=20.
# Booking 3 starts at 10 < 20 -> must open a NEW court.
assert courts_used(result_k2) >= 2

# Optimality sanity check: for random-ish disjoint-then-overlapping sets, min courts == max active overlap
def max_overlap(records):
    events = []
    for b in records:
        events.append((b.Start_time, 1))
        events.append((b.Finish_time, -1))
    events.sort(key=lambda e: (e[0], e[1]))    # ENDS before starts at same timestamp -- touching bookings don't overlap
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

for test_set in (bookings, back_to_back, triple_overlap, four_sequential):
    assert courts_used(assignCourts(test_set)) == max_overlap(test_set), [b.Id for b in test_set]

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Priority bookings needing better-amenity courts.** Partition courts into tiers and run the same greedy algorithm independently within each tier, routing higher-priority bookings to the better tier first; a booking that can't fit its preferred tier falls back to a lower one (a more complex bipartite-matching-flavored variant if amenity requirements can overlap in complex ways).
- **Cancellations after the initial assignment.** With a plain heap, removing an arbitrary booking's effect requires either **lazy deletion** (mark it cancelled, skip it when its heap entry surfaces, same trade-off discussed for the Content Popularity Tracker) or rebuilding the schedule from the remaining bookings.
- **Fixed number of courts -- reject bookings to maximize revenue.** This becomes weighted interval scheduling across multiple machines/resources, which is NP-hard in general (unlike the unlimited-courts version, which is solvable greedily in polynomial time) -- worth explicitly naming the complexity jump rather than trying to patch the greedy algorithm.
- **O(n) instead of O(n log n), given bounded time values.** If start/finish times are bounded integers in a known small range, bucket/counting sort replaces the O(n log n) comparison sort, and a bounded-size array can replace the heap for tracking availability -- trades generality for speed.
- **Recurring bookings (e.g. weekly lessons).** Expand each recurrence into its individual concrete `(start, finish)` instances up front (over whatever time horizon you're scheduling), then run the same algorithm -- recurrence expansion is a preprocessing step, not an algorithmic change.


## Empirical complexity check

All three variants are O(n log n) (one sort plus O(log n) heap work per booking). Doubling n should scale time by a bit more than 2x.

| Growth when n doubles | Implies |
|---|---|
| ~2x (or slightly more) | O(n log n) |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def make_worst_case(n):
    # Heavily overlapping bookings (worst case: every booking start falls inside many
    # earlier bookings' windows, maximizing heap churn) -- length-50 windows, dense starts.
    records = [make(i, i, i + 50) for i in range(n)]
    return (records,)


def run_basic(records):
    assignCourts(records)


def run_maintenance(records):
    assignCourtsWithMaintenance(records, X=5)


def run_threshold(records):
    assignCourtsWithThresholdMaintenance(records, X=5, K=3)


solutions = {
    "assignCourts": run_basic,
    "assignCourtsWithMaintenance": run_maintenance,
    "assignCourtsWithThresholdMaintenance": run_threshold,
}
sizes = [4000, 8000, 16000, 32000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Interval partitioning: min resources needed == max simultaneous overlap.** This lower bound is provable (any instant with k overlapping intervals needs k resources) and a greedy earliest-available assignment achieves it exactly -- no cleverer algorithm can do better, and none is needed.
- **Min-heap of "when does this resource free up" is the standard tool for greedy resource reuse.** The same shape (heap of end/availability times, reuse the earliest) appears in Meeting Rooms II, CPU task scheduling, and load balancing across servers.
- **State and prove the greedy choice with an exchange argument, don't just assert it.** "Picking the earliest-finishing option can always be swapped into any optimal solution without making it worse" is a one-line proof sketch that turns "this greedy algorithm happens to work" into "this greedy algorithm is provably optimal."
- **A parameterized side-effect (maintenance time, threshold count) usually only changes what gets pushed onto the heap, not the core algorithm.** Parts 2 and 3 both reused Part 1's control flow entirely -- only the computed `available_time` value differed.
- **Related problems:** Meeting Rooms II (LeetCode 253, identical "minimum rooms = max overlap" shape), Car Pooling (capacity-bounded overlap counting), Task Scheduler (CPU cooldown period, structurally similar to the maintenance-time follow-up).
- **Common pitfalls:** using `<` instead of `<=` when checking court availability (needlessly opens extra courts for legitimately back-to-back bookings); returning only a court *count* when the problem asks for the actual assignment; forgetting that maintenance-after-K needs per-court state (a plain global counter would be wrong -- each court's usage count is independent).
